精简版算法推荐

    逻辑回归

        核心价值：线性模型基石，输出概率解释性强，适合基线模型搭建

        适用场景：特征与目标呈近似线性关系、需模型可解释性的场景

    随机森林 (RF)

        核心价值：集成学习代表，抗过噪能力强，无需复杂调参

        适用场景：高维数据、缺失值较多、特征交互复杂的分类/回归问题

    XGBoost / LightGBM (二选一)

        核心价值：梯度提升标杆，预测精度高，适合竞赛与生产环境

        选择建议：

            优先 LightGBM：训练速度更快，内存占用低，适合大数据

            可选 XGBoost：理论更严谨，社区资源丰富，适合深入研究

    支持向量机 (SVM)

        核心价值：小样本高维数据表现优异，核技巧解决非线性问题

        适用场景：文本分类、图像识别等特征维度远高于样本量的场景

In [ ]:
# Core Scikit-learn 1.4+ 兼容导入
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import GridSearchCV

# 梯度提升库（注意 CatBoost 与 scikit-learn 的交互方式）
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 显式声明需要 joblib（旧版 sklearn.externals.joblib 已弃用）
import joblib  # 用于 CatBoost 等模型的并行化

In [ ]:
# Classifiers
classifiers = {
    "LogisticRegression" : LogisticRegression(random_state=0),
    "KNN" : KNeighborsClassifier(),
    "SVC" : SVC(random_state=0, probability=True),
    "RandomForest" : RandomForestClassifier(random_state=0),
    #"XGBoost" : XGBClassifier(random_state=0, use_label_encoder=False, eval_metric='logloss'), # XGBoost takes too long
    "LGBM" : LGBMClassifier(random_state=0),
    "CatBoost" : CatBoostClassifier(random_state=0, verbose=False),
    "NaiveBayes": GaussianNB()
}

# Grids for grid search

# LR_grid = {'penalty': ['l1','l2'],
#            'C': [0.25, 0.5, 0.75, 1, 1.25, 1.5],
#            'max_iter': [50, 100, 150]}

LR_grid = {
    'penalty': ['l2'],       # 仅保留兼容性更好的 l2 正则化
    'solver': ['lbfgs', 'sag'],  # 显式声明支持的 solver
    'C': [0.25, 0.5, 0.75, 1, 1.25, 1.5],
    'max_iter': [500]        # 新版建议增大迭代次数（原值 50-150 可能不足）
}

KNN_grid = {'n_neighbors': [3, 5, 7, 9],
            'p': [1, 2]}

SVC_grid = {'C': [0.25, 0.5, 0.75, 1, 1.25, 1.5],
            'kernel': ['linear', 'rbf'],
            'gamma': ['scale', 'auto']}

RF_grid = {'n_estimators': [50, 100, 150, 200, 250, 300],
        'max_depth': [4, 6, 8, 10, 12]}

boosted_grid = {'n_estimators': [50, 100, 150, 200],
        'max_depth': [4, 8, 12],
        'learning_rate': [0.05, 0.1, 0.15]}

NB_grid={'var_smoothing': [1e-10, 1e-9, 1e-8, 1e-7]}

# Dictionary of all grids
grid = {
    "LogisticRegression" : LR_grid,
    "KNN" : KNN_grid,
    "SVC" : SVC_grid,
    "RandomForest" : RF_grid,
    "XGBoost" : boosted_grid,
    "LGBM" : boosted_grid,
    "CatBoost" : boosted_grid,
    "NaiveBayes": NB_grid
}

**Train and evaluate models**

In [ ]:
i=0
clf_best_params=classifiers.copy()
valid_scores=pd.DataFrame({'Classifer':classifiers.keys(), 'Validation accuracy': np.zeros(len(classifiers)), 'Training time': np.zeros(len(classifiers))})
for key, classifier in classifiers.items():
    start = time.time()
    clf = GridSearchCV(estimator=classifier, param_grid=grid[key], n_jobs=-1, cv=None)

    # Train and score
    clf.fit(X_train, y_train)
    valid_scores.iloc[i,1]=clf.score(X_valid, y_valid)

    # Save trained model
    clf_best_params[key]=clf.best_params_
    
    # Print iteration and training time
    stop = time.time()
    valid_scores.iloc[i,2]=np.round((stop - start)/60, 2)
    
    print('Model:', key)
    print('Training time (mins):', valid_scores.iloc[i,2])
    print('')
    i+=1

In [ ]:
# Show results
valid_scores

In [ ]:
# Show best parameters from grid search
clf_best_params

**Define best models**

In [ ]:
# Classifiers
best_classifiers = {
    "LGBM" : LGBMClassifier(**clf_best_params["LGBM"], random_state=0),
    "CatBoost" : CatBoostClassifier(**clf_best_params["CatBoost"], verbose=False, random_state=0),
}

**Cross validation and ensembling predictions**

Predictions are ensembled together using soft voting. This averages the predicted probabilies to produce the most confident predictions.

In [ ]:
# Number of folds in cross validation
FOLDS=10

preds=np.zeros(len(X_test))
for key, classifier in best_classifiers.items():
    start = time.time()
    
    # 10-fold cross validation
    cv = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=0)
    
    score=0
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        # Get training and validation sets
        X_train, X_valid = X[train_idx], X[val_idx]
        y_train, y_valid = y[train_idx], y[val_idx]

        # Train model
        clf = classifier
        clf.fit(X_train, y_train)

        # Make predictions and measure accuracy
        preds += clf.predict_proba(X_test)[:,1]
        score += clf.score(X_valid, y_valid)

    # Average accuracy    
    score=score/FOLDS
    
    # Stop timer
    stop = time.time()

    # Print accuracy and time
    print('Model:', key)
    print('Average validation accuracy:', np.round(100*score,2))
    print('Training time (mins):', np.round((stop - start)/60,2))
    print('')
    
# Ensemble predictions
preds=preds/(FOLDS*len(best_classifiers))